# Hình 5 — ECG phục hồi thực tế từ checkpoint U4 MIX7–512

Notebook **suy luận, không huấn luyện**. Tự chứa đúng kiến trúc HNF–FiLM U-Net, diffusion/loss và sampler từ notebook U4; **không phải tải `src`**. Chỉ cần checkpoint `best_u4_mix7_protocol1_mixed512_100ep_v1.pth`.

**Colab:** bật GPU → Run all → chọn checkpoint nếu được hỏi → xem hình ngay ở mục 7–8. Nếu file nằm trong Drive, mount Drive và sửa `CHECKPOINT_PATH` ở mục 2. Không tự tải ảnh xuống.

**Phạm vi mặc định:** kiểm tra định tính nhỏ, 8 đoạn hợp lệ đầu tiên từ mỗi record `sel123`, `sel233`, tạo đủ 7 nhóm MIX7, tổng tối đa **112 cặp**. Đây **không phải đánh giá toàn bộ tập test**, không dùng các số trung bình này làm kết quả benchmark cuối. Chọn mẫu trước suy luận, không lựa theo chất lượng mô hình.

Đây là tín hiệu QTDB thực + artifact NSTDB cộng có kiểm soát, không phải nhiễu lâm sàng tự nhiên. Dữ liệu được tải trực tiếp từ PhysioNet và cache trong phiên; cần Internet. Không sử dụng `.npz` benchmark 3600 điểm.


## 1. Cài thư viện còn thiếu (giữ PyTorch có sẵn của Colab)

In [ ]:
import importlib.util
import subprocess
import sys

requirements = {'wfdb': 'wfdb>=4.1,<5', 'scipy': 'scipy>=1.10',
                'pandas': 'pandas>=2.0', 'matplotlib': 'matplotlib>=3.7', 'tqdm': 'tqdm>=4.62'}
missing = [package for module, package in requirements.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
print('Thư viện sẵn sàng; không cài lại torch.')


## 2. Tham số của lần thử

- `WEIGHTS_KIND='ema'`: dùng EMA **tại chính epoch được chọn bằng validation của mạng raw**. Có thể chọn `raw`, nhưng không đổi qua lại theo kết quả test để chọn bản đẹp hơn.
- `NOISE_LEVEL_R=1.0`: tỉ lệ peak-to-peak của **tổng nhiễu**, **không phải SNR 1 dB**. Cộng các patch trước, rồi scale tổng đúng một lần.
- Protocol 1: dùng **nửa sau NSTDB, kênh chỉ số 1**; train dùng nửa đầu, kênh chỉ số 0.
- Giữ sampler quadratic-tau nguyên bản U4, DDIM 50 bước, eta 0, 1 shot. Cố định seed, thứ tự mẫu, batch; khác thiết bị/phiên bản có thể sai khác số học.
- `PREVIEW_SAMPLE_ID=None`: lấy mẫu clean đầu tiên đã chọn, dùng cùng mẫu cho các nhóm nhiễu. Có thể nhập mã cụ thể sau khi xem manifest.


In [ ]:
import hashlib
import json
import math
import platform
import random
import time
from io import BytesIO
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import signal
import torch
import wfdb
from IPython.display import Image, display

CHECKPOINT_PATH = 'best_u4_mix7_protocol1_mixed512_100ep_v1.pth'
# Nếu dùng Drive: mount thủ công rồi thay CHECKPOINT_PATH bằng đường dẫn đầy đủ.
WEIGHTS_KIND = 'ema'  # 'ema' hoặc 'raw'; không tự fallback
TEST_RECORD_NAMES = ('sel123', 'sel233')
MAX_BEATS_PER_RECORD = 8
NOISE_GROUPS = ('BW', 'EM', 'MA', 'BW+EM', 'BW+MA', 'EM+MA', 'BW+EM+MA')
NOISE_LEVEL_R = 1.0
DDIM_STEPS, DDIM_ETA, NUM_SHOTS = 50, 0.0, 1
BATCH_SIZE = 8
INFERENCE_SEED = 1234
PREVIEW_SAMPLE_ID = None
FIGURE_GROUPS = ('BW', 'EM', 'MA', 'BW+EM+MA')
PREVIEW_DPI = 150
GRAYSCALE = False
CACHE_DIR = Path('/content/fig05_cache') if Path('/content').exists() else Path('output/fig05_cache')
EXPORT_DIR = Path('/content/fig05_output') if Path('/content').exists() else Path('output/fig05_output')

TARGET_FS, LENGTH, MAX_UNPADDED, INSERT_OFFSET = 360, 512, 496, 16
PHYSIONET_QT, PHYSIONET_NST = 'qtdb/1.0.0', 'nstdb/1.0.0'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(min(4, torch.get_num_threads()))
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
print('Thiết bị:', DEVICE)
if DEVICE.type != 'cuda':
    print('CPU vẫn chạy được, nhưng nên bật GPU trong Runtime > Change runtime type.')


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def sha_array(values):
    values = np.ascontiguousarray(values)
    digest = hashlib.sha256()
    digest.update(str(values.shape).encode('ascii'))
    digest.update(str(values.dtype).encode('ascii'))
    digest.update(values.tobytes())
    return digest.hexdigest()


def wfdb_read_with_retry(operation, *args, **kwargs):
    for attempt in range(5):
        try:
            return operation(*args, **kwargs)
        except Exception as exc:
            transient = any(token in str(exc).lower() for token in (
                '429', '500 error', '502', '503', '504', 'bad gateway',
                'timeout', 'timed out', 'connection error', 'connection reset',
                'temporarily unavailable',
            ))
            if not transient or attempt == 4:
                raise
            delay = min(2 ** (attempt + 1), 16)
            print(f'PhysioNet: retry {attempt + 1}/4 sau {delay}s: {exc}')
            time.sleep(delay)


def resolve_checkpoint(path):
    path = Path(path)
    if path.is_file():
        return path
    local = Path('checkpoints/mix') / path.name
    if local.is_file():
        return local
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError(f'Không thấy checkpoint: {path}') from None
    print('Chọn best_u4_mix7_protocol1_mixed512_100ep_v1.pth; không cần tải src.')
    uploaded = files.upload()
    if path.is_file():
        return path
    matches = [Path(name) for name in uploaded if Path(name).name == path.name]
    if len(matches) != 1:
        raise FileNotFoundError(f'Cần đúng checkpoint {path.name}; sửa CHECKPOINT_PATH nếu đổi tên.')
    return matches[0]

## 3. Kiến trúc và diffusion nguyên bản (đã nhúng)

Khối HNF, FiLM, attention, U-Net và các buffer diffusion giữ nguyên từ `kaggle_ablation_u4_all_noise_512_100ep.ipynb`. Kiểm tra state dict bằng `strict=True`; không bỏ qua trọng số thiếu/thừa. Mã loss được giữ để tương thích lớp checkpoint, nhưng notebook này **không gọi huấn luyện**.


In [ ]:
import torch.nn as nn


class HNFBlock(nn.Module):
    """
    Half Normalized Filter block.
    Multi-scale convs → concat → conv → half‑instance norm → residual.
    """
    def __init__(self, in_channels, out_channels, kernel_sizes=(3, 5, 9, 15)):
        super().__init__()
        self.multi_convs = nn.ModuleList([
            nn.Conv1d(in_channels, out_channels // len(kernel_sizes), k,
                      padding=k//2, padding_mode='reflect')
            for k in kernel_sizes
        ])
        self.agg_conv = nn.Conv1d(out_channels, out_channels, 1)
        # Half‑instance norm: chia channel làm 2 nửa
        self.half_inst_norm = nn.InstanceNorm1d(out_channels // 2)
        self.act = nn.ReLU(inplace=True)
        self.residual = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else nn.Identity()

    def forward(self, x):
        multi_out = [conv(x) for conv in self.multi_convs]
        out = torch.cat(multi_out, dim=1)
        out = self.agg_conv(out)
        half = out.shape[1] // 2
        out_norm = self.half_inst_norm(out[:, :half, :])
        out = torch.cat([out_norm, out[:, half:, :]], dim=1)
        out = self.act(out)
        return out + self.residual(x)


class BridgeBlock(nn.Module):
    """FiLM‑based conditioning on noise level (sqrt(alpha_bar))."""
    def __init__(self, features, emb_dim=128):
        super().__init__()
        self.emb_dim = emb_dim
        self.film = nn.Sequential(
            nn.Linear(emb_dim, features * 2),
            nn.SiLU()
        )

    def forward(self, x, alpha_bar):
        # alpha_bar: (B,) or scalar
        # Ép phẳng alpha_bar về dạng 1D (Batch,) để tránh bị dư chiều -- CỰC QUAN TRỌNG
        alpha_bar = alpha_bar.view(-1)
        emb = self.sinusoidal_embedding(alpha_bar)
        scale_shift = self.film(emb)   # (B, 2*features)
        scale, shift = scale_shift.chunk(2, dim=1)
        scale = scale.unsqueeze(-1)
        shift = shift.unsqueeze(-1)
        return x * (1 + scale) + shift

    def sinusoidal_embedding(self, x):
        if x.dim() == 0:
            x = x.unsqueeze(0)
        device = x.device
        half_dim = self.emb_dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = x.unsqueeze(-1) * emb.unsqueeze(0)
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        return emb


class SelfAttention1D(nn.Module):
    """1D Self‑Attention (multi‑head) dùng tại bottleneck."""
    def __init__(self, channels, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = channels // num_heads
        assert self.head_dim * num_heads == channels, "channels phải chia hết cho num_heads"

        self.qkv = nn.Conv1d(channels, channels * 3, kernel_size=1)
        self.proj = nn.Conv1d(channels, channels, kernel_size=1)

    def forward(self, x):
        B, C, L = x.shape
        qkv = self.qkv(x).reshape(B, 3, self.num_heads, self.head_dim, L)
        q, k, v = qkv[:, 0], qkv[:, 1], qkv[:, 2]
        # q, k, v: (B, num_heads, head_dim, L)
        attn = torch.matmul(q.transpose(-2, -1), k) / (self.head_dim ** 0.5)
        attn = torch.softmax(attn, dim=-1)
        out = torch.matmul(attn, v.transpose(-2, -1)).transpose(-2, -1)
        out = out.reshape(B, C, L)
        return self.proj(out)


class UNet1D(nn.Module):
    """U4: same four-level conditional diffusion U-Net scaffold as U4."""

    def __init__(self, in_channels=2, base_channels=80, emb_dim=128,
                 out_channels=1):
        super().__init__()
        c = base_channels
        self.enc1 = HNFBlock(in_channels, c)
        self.bridge1 = BridgeBlock(c, emb_dim)
        self.down1 = nn.Conv1d(c, c * 2, 4, stride=2, padding=1)
        self.enc2 = HNFBlock(c * 2, c * 2)
        self.bridge2 = BridgeBlock(c * 2, emb_dim)
        self.down2 = nn.Conv1d(c * 2, c * 4, 4, stride=2, padding=1)
        self.enc3 = HNFBlock(c * 4, c * 4)
        self.bridge3 = BridgeBlock(c * 4, emb_dim)
        self.down3 = nn.Conv1d(c * 4, c * 8, 4, stride=2, padding=1)
        self.enc4 = HNFBlock(c * 8, c * 8)
        self.bridge4 = BridgeBlock(c * 8, emb_dim)
        self.attn = SelfAttention1D(c * 8)
        self.up4 = nn.ConvTranspose1d(c * 8, c * 4, 4, stride=2, padding=1)
        self.dec4 = HNFBlock(c * 8, c * 4)
        self.up3 = nn.ConvTranspose1d(c * 4, c * 2, 4, stride=2, padding=1)
        self.dec3 = HNFBlock(c * 4, c * 2)
        self.up2 = nn.ConvTranspose1d(c * 2, c, 4, stride=2, padding=1)
        self.dec2 = HNFBlock(c * 2, c)
        self.final = nn.Conv1d(c, out_channels, 1)

    def forward(self, x, cond, noise_scale):
        if x.shape != cond.shape or x.shape[-1] % 8:
            raise ValueError('Expected matching [B,1,L] inputs with L divisible by 8')
        e1 = self.bridge1(self.enc1(torch.cat([x, cond], dim=1)), noise_scale)
        e2 = self.bridge2(self.enc2(self.down1(e1)), noise_scale)
        e3 = self.bridge3(self.enc3(self.down2(e2)), noise_scale)
        e4 = self.attn(self.bridge4(self.enc4(self.down3(e3)), noise_scale))
        d4 = self.dec4(torch.cat([self.up4(e4), e3], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e2], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e1], dim=1))
        return self.final(d2)

In [ ]:
from functools import partial
from inspect import isfunction
from tqdm.auto import tqdm
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from functools import partial
from inspect import isfunction

# ----- Bắt buộc phải có torchaudio hoặc dùng torch.stft -----
# Nếu không có torchaudio, ta dùng torch.stft nguyên bản
def stft_loss(pred, target, n_fft=128, hop_length=64):
    """
    Tính MSE loss trên phổ STFT giữa pred và target.
    """
    # (B, C, L) -> (B*C, L)
    B, C, L = pred.shape
    pred = pred.view(-1, L)
    target = target.view(-1, L)
    
    # STFT: returns (B, freq_bins, time_frames, 2)
    spec_pred = torch.stft(pred, n_fft=n_fft, hop_length=hop_length, 
                           return_complex=True)
    spec_target = torch.stft(target, n_fft=n_fft, hop_length=hop_length,
                             return_complex=True)
    
    # Lấy magnitude (độ lớn)
    mag_pred = torch.abs(spec_pred)
    mag_target = torch.abs(spec_target)
    
    return F.mse_loss(mag_pred, mag_target)


def exists(x):
    return x is not None

def default(val, d):
    if exists(val):
        return val
    return d() if isfunction(d) else d


class DDPM(nn.Module):
    def __init__(self, base_model, config, device, conditional=True):
        super().__init__()
        self.device = device
        self.model = base_model
        self.config = config
        self.conditional = conditional
        
        # ----------------------
        # 👉 THÊM: Loss weights
        # ----------------------
        self.lambda_time = config['train'].get('lambda_time', 1.0)
        self.lambda_freq = config['train'].get('lambda_freq', 0.1)  # cân bằng
        
        self.loss_func_l1 = nn.L1Loss(reduction='sum').to(device)
        
        config_diff = config["diffusion"]
        self.num_steps = config_diff["num_steps"]
        self.set_new_noise_schedule(config_diff, device)
        
    def make_beta_schedule(self, schedule='linear', n_timesteps=1000, start=1e-5, end=1e-2):
        if schedule == 'linear':
            betas = torch.linspace(start, end, n_timesteps)
        elif schedule == "quad":
            betas = torch.linspace(start ** 0.5, end ** 0.5, n_timesteps) ** 2
        elif schedule == "sigmoid":
            betas = torch.linspace(-6, 6, n_timesteps)
            betas = torch.sigmoid(betas) * (end - start) + start
        return betas
    
    def set_new_noise_schedule(self, config_diff, device):
        to_torch = partial(torch.tensor, dtype=torch.float32, device=device)
        
        betas = self.make_beta_schedule(
            schedule=config_diff["schedule"], 
            n_timesteps=config_diff["num_steps"],
            start=config_diff["beta_start"], 
            end=config_diff["beta_end"]
        )
        betas = betas.detach().cpu().numpy() if isinstance(betas, torch.Tensor) else betas
        
        alphas = 1. - betas
        alphas_cumprod = np.cumprod(alphas, axis=0)
        alphas_cumprod_prev = np.append(1., alphas_cumprod[:-1])
        self.sqrt_alphas_cumprod_prev = np.sqrt(np.append(1., alphas_cumprod))
        
        self.register_buffer('betas', to_torch(betas))
        self.register_buffer('alphas_cumprod', to_torch(alphas_cumprod))
        self.register_buffer('alphas_cumprod_prev', to_torch(alphas_cumprod_prev))
        self.register_buffer('sqrt_alphas_cumprod', to_torch(np.sqrt(alphas_cumprod)))
        self.register_buffer('sqrt_one_minus_alphas_cumprod', to_torch(np.sqrt(1. - alphas_cumprod)))
        self.register_buffer('log_one_minus_alphas_cumprod', to_torch(np.log(1. - alphas_cumprod)))
        self.register_buffer('sqrt_recip_alphas_cumprod', to_torch(np.sqrt(1. / alphas_cumprod)))
        self.register_buffer('sqrt_recipm1_alphas_cumprod', to_torch(np.sqrt(1. / alphas_cumprod - 1)))
        
        posterior_variance = betas * (1. - alphas_cumprod_prev) / (1. - alphas_cumprod)
        self.register_buffer('posterior_variance', to_torch(posterior_variance))
        self.register_buffer('posterior_log_variance_clipped', 
                             to_torch(np.log(np.maximum(posterior_variance, 1e-20))))
        self.register_buffer('posterior_mean_coef1', 
                             to_torch(betas * np.sqrt(alphas_cumprod_prev) / (1. - alphas_cumprod)))
        self.register_buffer('posterior_mean_coef2', 
                             to_torch((1. - alphas_cumprod_prev) * np.sqrt(alphas) / (1. - alphas_cumprod)))
        
    def predict_start_from_noise(self, x_t, t, noise):
        # Nắn lại chiều tensor hệ số thành (Batch, 1, 1) để nhân đúng với tensor tín hiệu 3D
        coef1 = self.sqrt_recip_alphas_cumprod[t].view(-1, 1, 1)
        coef2 = self.sqrt_recipm1_alphas_cumprod[t].view(-1, 1, 1)
        
        return coef1 * x_t - coef2 * noise
    
    def q_posterior(self, x_start, x_t, t):
        posterior_mean = self.posterior_mean_coef1[t] * x_start + \
                         self.posterior_mean_coef2[t] * x_t
        posterior_log_variance_clipped = self.posterior_log_variance_clipped[t]
        return posterior_mean, posterior_log_variance_clipped
    
    def p_mean_variance(self, x, t, clip_denoised: bool, condition_x=None):
        batch_size = x.shape[0]
        noise_level = torch.FloatTensor(
            [self.sqrt_alphas_cumprod_prev[t+1]]
        ).repeat(batch_size, 1).to(x.device)
        if condition_x is not None:
            x_recon = self.predict_start_from_noise(
                x, t=t, noise=self.model(x, condition_x, noise_level))
        else:
            x_recon = self.predict_start_from_noise(
                x, t=t, noise=self.model(x, noise_level))
        if clip_denoised:
            x_recon.clamp_(-1., 1.)
        model_mean, posterior_log_variance = self.q_posterior(
            x_start=x_recon, x_t=x, t=t)
        return model_mean, posterior_log_variance
    
    @torch.no_grad()
    def p_sample(self, x, t, clip_denoised=False, condition_x=None):
        model_mean, model_log_variance = self.p_mean_variance(
            x=x, t=t, clip_denoised=clip_denoised, condition_x=condition_x)
        noise = torch.randn_like(x) if t > 0 else torch.zeros_like(x)
        return model_mean + noise * (0.5 * model_log_variance).exp()

    @torch.no_grad()
    def p_sample_loop(self, x_in, continous=False):
        device = self.betas.device
        sample_inter = (1 | (self.num_steps//10))
        if not self.conditional:
            shape = x_in
            cur_x = torch.randn(shape, device=device)
            ret_x = cur_x
            for i in reversed(range(0, self.num_steps)):
                cur_x = self.p_sample(cur_x, i)
                if i % sample_inter == 0:
                    ret_x = torch.cat([ret_x, cur_x], dim=0)
        else:
            x = x_in
            shape = x.shape
            cur_x = torch.randn(shape, device=device)
            ret_x = [cur_x]
            for i in reversed(range(0, self.num_steps)):
                cur_x = self.p_sample(cur_x, i, condition_x=x)
                if i % sample_inter == 0:
                    ret_x.append(cur_x)
        if continous:
            return ret_x
        else:
            return ret_x[-1]    
    
    @torch.no_grad()
    def sample(self, batch_size=1, shape=[1, 512], continous=False):
        return self.p_sample_loop((batch_size, shape[0], shape[1]), continous)
    
    # ==========================================================
    # 👉 DDIM SAMPLER (đã có sẵn trong source gốc)
    # ==========================================================
    @torch.no_grad()
    def ddim_sample_loop(self, x_in, ddim_timesteps=50, ddim_eta=0.0, num_shots=1):
        device = self.betas.device
        T = self.num_steps
        S = ddim_timesteps
        tau = [int(np.floor((T / (S ** 2)) * (i ** 2))) for i in range(S + 1)]
        
        alphas_cumprod_with_1 = torch.cat([
            torch.tensor([1.0], dtype=self.alphas_cumprod.dtype, device=device),
            self.alphas_cumprod
        ]).float()
        
        out_accum = 0.0
        for shot in range(num_shots):
            shape = x_in.shape
            x = torch.randn(shape, device=device)
            for i in reversed(range(1, S + 1)):
                t = tau[i]
                t_prev = tau[i - 1]
                noise_level = torch.FloatTensor(
                    [self.sqrt_alphas_cumprod_prev[t]]
                ).repeat(x.shape[0], 1).to(device)
                if not self.conditional:
                    eps = self.model(x, noise_level)
                else:
                    eps = self.model(x, x_in, noise_level)
                    
                a_t = alphas_cumprod_with_1[t]
                a_t_prev = alphas_cumprod_with_1[t_prev]
                x0_pred = (x - torch.sqrt(1. - a_t) * eps) / torch.sqrt(a_t)
                if t_prev == 0:
                    sigma_t = 0.0
                else:
                    sigma_t = ddim_eta * torch.sqrt((1. - a_t_prev) / (1. - a_t)) * \
                              torch.sqrt(torch.clamp(1. - a_t / a_t_prev, min=0.0))
                direction = torch.sqrt(torch.clamp(1. - a_t_prev - sigma_t**2, min=0.0)) * eps
                noise = sigma_t * torch.randn_like(x) if (ddim_eta > 0 and t_prev > 0) else 0.0
                x = torch.sqrt(a_t_prev) * x0_pred + direction + noise
            out_accum += x
        return out_accum / num_shots

    @torch.no_grad()
    def denoising(self, x_in, continous=False, use_ddim=False, ddim_steps=50, ddim_eta=0.0, num_shots=1):
        if use_ddim:
            return self.ddim_sample_loop(x_in, ddim_timesteps=ddim_steps, ddim_eta=ddim_eta, num_shots=num_shots)
        else:
            if num_shots > 1:
                out_accum = 0.0
                for _ in range(num_shots):
                    out_accum += self.p_sample_loop(x_in, continous)
                return out_accum / num_shots
            return self.p_sample_loop(x_in, continous)
    
    def q_sample_loop(self, x_start, continous=False):
        sample_inter = (1 | (self.num_steps//10))
        ret_x = [x_start]
        cur_x = x_start
        for t in range(1, self.num_steps+1):
            B,C,L = cur_x.shape
            continuous_sqrt_alpha_cumprod = torch.FloatTensor(
                np.random.uniform(
                    self.sqrt_alphas_cumprod_prev[t-1],
                    self.sqrt_alphas_cumprod_prev[t],
                    size=B
                )
            ).to(cur_x.device)
            continuous_sqrt_alpha_cumprod = continuous_sqrt_alpha_cumprod.view(B, -1)
            noise = torch.randn_like(cur_x)
            cur_x = self.q_sample(
                x_start=cur_x, 
                continuous_sqrt_alpha_cumprod=continuous_sqrt_alpha_cumprod.view(-1, 1, 1), 
                noise=noise
            )
            if t % sample_inter == 0:
                ret_x.append(cur_x)
        if continous:
            return ret_x
        else:
            return ret_x[-1]
    
    def q_sample(self, x_start, continuous_sqrt_alpha_cumprod, noise=None):
        noise = default(noise, lambda: torch.randn_like(x_start))
        return continuous_sqrt_alpha_cumprod * x_start + \
               (1 - continuous_sqrt_alpha_cumprod**2).sqrt() * noise
    
    # ==========================================================
    # 👉 HÀM LOSS CHÍNH (ĐÃ THÊM STFT)
    # ==========================================================
    def p_losses(self, x_in, y_in, noise=None):
        """
        Multi‑domain Loss:
        - L1 trên noise (miền thời gian)
        - STFT Loss trên tín hiệu dự đoán (miền tần số)
        """
        x_start = x_in
        B, C, L = x_start.shape

        # Lấy ngẫu nhiên các giá trị t khác nhau cho từng mẫu trong Batch, trực tiếp trên thiết bị (GPU)
        t = torch.randint(0, self.num_steps, (B,), device=x_start.device).long()
        # 1. Chuyển mảng NumPy sang PyTorch Tensor và đẩy lên cùng GPU với t
        sqrt_alphas_prev_tensor = torch.tensor(self.sqrt_alphas_cumprod_prev, dtype=torch.float32, device=t.device)
        # 2. Lấy giới hạn dưới và trên. 
        # (Lưu ý: Do t giờ chạy từ 0 đến num_steps-1, ta dùng chỉ số t và t+1 để thay thế cho t-1 và t nhằm tránh lỗi âm)
        lower = sqrt_alphas_prev_tensor[t]
        upper = sqrt_alphas_prev_tensor[t+1]
        # 3. Sinh số ngẫu nhiên uniform hoàn toàn bằng PyTorch trực tiếp trên GPU
        continuous_sqrt_alpha_cumprod = lower + torch.rand(B, device=t.device) * (upper - lower)
        continuous_sqrt_alpha_cumprod = continuous_sqrt_alpha_cumprod.view(B, -1)
        noise = default(noise, lambda: torch.randn_like(x_start))
        x_noisy = self.q_sample(
            x_start=x_start, 
            continuous_sqrt_alpha_cumprod=continuous_sqrt_alpha_cumprod.view(-1, 1, 1), 
            noise=noise
        )

        # Dự đoán noise
        if not self.conditional:
            x_recon = self.model(x_noisy, continuous_sqrt_alpha_cumprod)
        else:
            x_recon = self.model(x_noisy, y_in, continuous_sqrt_alpha_cumprod)

        # ---------- 1. L1 Loss (miền thời gian) ----------
        loss_l1 = self.loss_func_l1(noise, x_recon)  # (B*C*L)

        # ---------- 2. STFT Loss (miền tần số) ----------
        # Dự đoán x0 từ noise đã dự đoán
        x0_pred = self.predict_start_from_noise(x_noisy, t, x_recon)
        
        # Tính STFT loss giữa x0_pred và x_start (clean gốc)
        loss_stft = stft_loss(x0_pred, x_start, n_fft=128, hop_length=64)
        
        # ---------- 3. Tổng hợp ----------
        total_loss = self.lambda_time * loss_l1 + self.lambda_freq * loss_stft
        return total_loss
    
    def forward(self, x, y, *args, **kwargs):
        return self.p_losses(x, y, *args, **kwargs)


class EMA(object):
    def __init__(self, mu=0.999):
        self.mu = mu
        self.shadow = {}

    def register(self, module):
        for name, param in module.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self, module):
        for name, param in module.named_parameters():
            if param.requires_grad:
                self.shadow[name].data = (1. - self.mu) * param.data + self.mu * self.shadow[name].data

    def ema(self, module):
        for name, param in module.named_parameters():
            if param.requires_grad:
                param.data.copy_(self.shadow[name].data)

    def ema_copy(self, module):
        module_copy = type(module)(module.config).to(module.config.device)
        module_copy.load_state_dict(module.state_dict())
        self.ema(module_copy)
        return module_copy

    def state_dict(self):
        return self.shadow

    def load_state_dict(self, state_dict):
        self.shadow = state_dict

class ResearchLossDDPM(DDPM):
    """Same DDPM/UNet architecture, with a scale-stable multi-domain objective."""

    def loss_components(self, x_in, y_in, noise=None):
        x_start = x_in
        batch = x_start.shape[0]
        timestep = torch.randint(0, self.num_steps, (batch,), device=x_start.device)
        boundaries = torch.as_tensor(
            self.sqrt_alphas_cumprod_prev, dtype=x_start.dtype, device=x_start.device
        )
        lower, upper = boundaries[timestep], boundaries[timestep + 1]
        continuous = lower + torch.rand(batch, device=x_start.device) * (upper - lower)
        continuous_2d = continuous.view(batch, 1)
        continuous_3d = continuous.view(batch, 1, 1)
        gaussian_noise = default(noise, lambda: torch.randn_like(x_start))
        x_noisy = self.q_sample(x_start, continuous_3d, gaussian_noise)
        predicted_noise = self.model(x_noisy, y_in, continuous_2d)

        # Mean keeps the time loss independent of batch size and 512-sample length.
        loss_time = F.l1_loss(predicted_noise, gaussian_noise, reduction='mean')

        # Reconstruct x0 with the exact continuous alpha used to create x_t.
        residual_scale = torch.sqrt(torch.clamp(1.0 - continuous_3d ** 2, min=0.0))
        x0_prediction = (x_noisy - residual_scale * predicted_noise) / continuous_3d.clamp_min(1e-6)

        n_fft, hop_length = 128, 64
        window = torch.hann_window(n_fft, dtype=x_start.dtype, device=x_start.device)
        pred_flat = x0_prediction.reshape(-1, x0_prediction.shape[-1])
        target_flat = x_start.reshape(-1, x_start.shape[-1])
        pred_mag = torch.abs(torch.stft(pred_flat, n_fft=n_fft, hop_length=hop_length,
                                         window=window, return_complex=True))
        target_mag = torch.abs(torch.stft(target_flat, n_fft=n_fft, hop_length=hop_length,
                                           window=window, return_complex=True))
        pred_mag = pred_mag.reshape(batch, -1)
        target_mag = target_mag.reshape(batch, -1)
        spectral_error = torch.mean((pred_mag - target_mag) ** 2, dim=1)
        target_power = torch.mean(target_mag ** 2, dim=1).clamp_min(1e-6)
        relative_frequency = spectral_error / target_power

        # x0 is ill-conditioned at high-noise timesteps, so weight STFT by diffusion SNR.
        alpha_bar = continuous ** 2
        diffusion_snr = alpha_bar / torch.clamp(1.0 - alpha_bar, min=1e-6)
        frequency_weight = torch.clamp(diffusion_snr, max=1.0)
        loss_frequency = torch.mean(relative_frequency * frequency_weight)
        total = self.lambda_time * loss_time + self.lambda_freq * loss_frequency
        return {'total': total, 'time': loss_time, 'frequency': loss_frequency,
                'frequency_unweighted': relative_frequency.mean(),
                'frequency_weight': frequency_weight.mean()}

    def p_losses(self, x_in, y_in, noise=None):
        return self.loss_components(x_in, y_in, noise)['total']


In [ ]:
def load_u4_checkpoint(path, weights_kind, device):
    # Only load the tensor/basic-container checkpoint format used by this project.
    payload = torch.load(path, map_location='cpu', weights_only=True)
    expected = {
        'ablation_id': 'U4', 'artifact': 'MIX7', 'protocol': 1,
    }
    for name, value in expected.items():
        if payload.get(name) != value:
            raise ValueError(f'Checkpoint không đúng {name}={value}: {payload.get(name)}')
    config = payload['config']
    checks = {
        'model_normalization': 'per_segment_clean_maxabs_shared_with_noisy_v1',
        'preprocessing_version': 'qtdb_pu1_p_and_beat_40ms_reflect100ms_360hz_endpoints_pad16_v2',
        'data_protocol': 'qtdb_nstdb_balanced_7_nonempty_masks_512_v1',
        'loss_version': 'l1_mean_relative_stft_snr_weighted_continuous_alpha_v1',
    }
    for name, value in checks.items():
        if config.get(name) != value:
            raise ValueError(f'Khác giao thức {name}: {config.get(name)}')
    if config.get('architecture') != {
        'id': 'U4', 'feature_block': 'hnf_multiscale_halfnorm_residual',
        'timestep': 'sinusoidal_film_scale_shift', 'attention': 'self_attention_4head',
    } or config['train']['feats'] != 80:
        raise ValueError('Cấu hình kiến trúc không phải U4 gốc.')
    if not set(payload['train_records']).isdisjoint(payload['test_records']):
        raise ValueError('Train/test records bị trùng.')
    if weights_kind not in ('ema', 'raw'):
        raise ValueError("WEIGHTS_KIND phải là 'ema' hoặc 'raw'")
    state_key = 'ema_model_state_dict' if weights_kind == 'ema' else 'state_dict'
    if state_key not in payload:
        raise KeyError(f'Checkpoint thiếu {state_key}; không tự chuyển loại trọng số.')
    model = ResearchLossDDPM(UNet1D(2, 80, 128, 1), config, device).to(device)
    model.load_state_dict(payload[state_key], strict=True)
    model.eval()
    count = sum(p.numel() for p in model.parameters())
    if count != payload['parameter_count']:
        raise ValueError('Số tham số không khớp checkpoint.')
    provenance = {
        'checkpoint_name': Path(path).name, 'checkpoint_sha256': sha256_file(path),
        'weights': weights_kind, 'checkpoint_epoch_one_based': int(payload['epoch']) + 1,
        'best_validation_loss_raw_model': float(payload['best_validation_loss']),
        'parameter_count': count, 'config': config,
        'train_records': list(payload['train_records']),
        'test_records': list(payload['test_records']),
        'training_data_fingerprint': payload['data_fingerprint'],
        'torch': torch.__version__, 'numpy': np.__version__,
        'wfdb': wfdb.__version__, 'python': platform.python_version(),
        'device': str(device),
    }
    print(f"U4 MIX7 | epoch {provenance['checkpoint_epoch_one_based']} | "
          f"{weights_kind.upper()} | {count:,} tham số | {device}")
    print('Best validation loss (raw model):', provenance['best_validation_loss_raw_model'])
    return model, provenance

checkpoint_path = resolve_checkpoint(CHECKPOINT_PATH)
MODEL, PROVENANCE = load_u4_checkpoint(checkpoint_path, WEIGHTS_KIND, DEVICE)


## 4. QTDB test và NSTDB giữ riêng

Tiền xử lý giống train: `.pu1`, mốc bắt đầu trước P 40 ms; resample có reflect padding 100 ms về 360 Hz; loại đoạn không hợp lệ; trừ trung bình hai đầu; chèn tối đa 496 điểm vào tensor 512 tại offset 16. Không kéo giãn từng nhịp thành 512 điểm.

Chỉ chọn record trong danh sách test của checkpoint và **không bỏ qua record khi tải lỗi**. Kiểm tra hash của toàn bộ BW/EM/MA với fingerprint lúc train. Cache tạo ở `CACHE_DIR`; nếu cache lỗi, chỉ xóa đúng file cache lỗi rồi chạy lại.


In [ ]:
BEAT_SYMBOLS = set('NLRBAaJSVrFejnE/fQ?')

def p_onsets(annotation):
    symbols = np.asarray(annotation.symbol)
    samples = np.asarray(annotation.sample, dtype=np.int64)
    return np.asarray([int(samples[i]) for i, symbol in enumerate(symbols)
                       if symbol == '(' and 'p' in symbols[i + 1:i + 4]], dtype=np.int64)

def r_locations(annotation):
    return np.asarray([int(sample) for sample, symbol in zip(annotation.sample, annotation.symbol)
                       if symbol in BEAT_SYMBOLS], dtype=np.int64)

def resample_reflect(segment, original_fs):
    pad = min(max(1, int(round(0.1 * original_fs))), len(segment) - 1)
    padded = np.pad(segment, (pad, pad), mode='reflect')
    new_length = int(round(len(padded) * TARGET_FS / original_fs))
    resampled = signal.resample(padded, new_length).astype(np.float32)
    target_pad = int(round(pad * TARGET_FS / original_fs))
    expected = int(round(len(segment) * TARGET_FS / original_fs))
    return resampled[target_pad:target_pad + expected]

def clean_segments_for_record(name):
    record = wfdb_read_with_retry(wfdb.rdrecord, name, pn_dir=PHYSIONET_QT, channels=[0])
    # QTDB does not provide .atr for every record (e.g. sel30).
    # .pu1 provides P boundaries and beat fiducials throughout each record.
    pu = wfdb_read_with_retry(wfdb.rdann, name, 'pu1', pn_dir=PHYSIONET_QT)
    fs = float(record.fs)
    ecg = record.p_signal[:, 0].astype(np.float32)
    starts = p_onsets(pu) - int(round(0.04 * fs))
    starts = starts[(starts >= 0) & (starts < len(ecg))]
    peaks = r_locations(pu)
    beats, rows = [], []
    audit = {'record_id': name, 'p_onsets': len(starts),
             'candidate_intervals': max(0, len(starts) - 1),
             'rejected_short_interval': 0, 'rejected_multiple_r': 0,
             'rejected_length': 0, 'rejected_nonfinite_or_flat': 0}
    resampled_lengths = []
    for segment_index, (start, end) in enumerate(zip(starts[:-1], starts[1:])):
        if end <= start + 2:
            audit['rejected_short_interval'] += 1
            continue
        r_count = int(np.count_nonzero((peaks >= start) & (peaks < end)))
        if r_count > 1:
            audit['rejected_multiple_r'] += 1
            continue
        segment = resample_reflect(ecg[start:end], fs)
        resampled_lengths.append(len(segment))
        if not 2 <= len(segment) <= MAX_UNPADDED:
            audit['rejected_length'] += 1
            continue
        segment = segment - 0.5 * (segment[0] + segment[-1])
        clean = np.zeros(LENGTH, dtype=np.float32)
        clean[INSERT_OFFSET:INSERT_OFFSET + len(segment)] = segment
        if not np.all(np.isfinite(clean)) or np.ptp(clean) <= 0:
            audit['rejected_nonfinite_or_flat'] += 1
            continue
        beats.append(clean)
        rows.append({'sample_id': f'{name}:{segment_index}', 'patient_id': name,
                     'record_id': name, 'clean_segment_id': f'{name}:{segment_index}',
                     'start_sample': int(start), 'end_sample': int(end),
                     'unpadded_length': len(segment), 'r_count': r_count,
                     'annotation_source': 'pu1'})
    audit['min_resampled_length'] = min(resampled_lengths) if resampled_lengths else np.nan
    audit['max_resampled_length'] = max(resampled_lengths) if resampled_lengths else np.nan
    audit['accepted_segments'] = len(beats)
    return beats, rows, audit



In [ ]:
def prepare_pilot_data(provenance, record_names, max_beats_per_record, cache_dir):
    if not record_names or len(set(record_names)) != len(record_names):
        raise ValueError('Cần các record test không trùng nhau.')
    if not set(record_names).issubset(provenance['test_records']):
        raise ValueError('Chỉ được chọn record test giữ riêng trong checkpoint.')
    if max_beats_per_record < 1:
        raise ValueError('MAX_BEATS_PER_RECORD phải >= 1.')
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    clean_parts, metadata, audits = [], [], []
    for record_name in record_names:
        cache = cache_dir / f'{record_name}_clean512_v2.npz'
        if cache.is_file():
            with np.load(cache, allow_pickle=False) as saved:
                beats = saved['clean']
                rows = json.loads(saved['metadata_json'].item())
                audit = json.loads(saved['audit_json'].item())
        else:
            # This is the same extraction function as in the U4 training notebook.
            beats, rows, audit = clean_segments_for_record(record_name)
            if not beats:
                raise RuntimeError(f'{record_name}: không có đoạn hợp lệ; không bỏ qua record.')
            beats = np.stack(beats).astype(np.float32)
            np.savez_compressed(cache, clean=beats, metadata_json=json.dumps(rows),
                                audit_json=json.dumps(audit))
        if beats.shape != (len(rows), LENGTH) or not np.isfinite(beats).all():
            raise ValueError(f'Cache clean không hợp lệ: {cache}')
        if any(row['record_id'] != record_name for row in rows):
            raise ValueError(f'Cache sai record: {cache}')
        take = min(max_beats_per_record, len(beats))
        # Fixed first accepted beats, selected BEFORE inference, not by model quality.
        clean_parts.append(beats[:take])
        metadata.extend(rows[:take])
        audits.append({**audit, 'selected_for_pilot': take})
        print(f'{record_name}: dùng {take}/{len(beats)} đoạn hợp lệ đầu tiên')
    clean = np.concatenate(clean_parts).astype(np.float32)
    meta = pd.DataFrame(metadata)
    if not meta.sample_id.is_unique:
        raise ValueError('sample_id clean bị trùng.')
    noise = {}
    for artifact in ('BW', 'EM', 'MA'):
        cache = cache_dir / f'{artifact.lower()}_360.npy'
        if cache.is_file():
            values = np.load(cache, allow_pickle=False)
        else:
            record = wfdb_read_with_retry(wfdb.rdrecord, artifact.lower(), pn_dir=PHYSIONET_NST)
            values = record.p_signal.astype(np.float32)
            if int(record.fs) != TARGET_FS:
                values = signal.resample(values, int(round(
                    len(values) * TARGET_FS / float(record.fs))), axis=0).astype(np.float32)
            np.save(cache, values)
        if values.ndim != 2 or values.shape[1] < 2 or not np.isfinite(values).all():
            raise ValueError(f'NSTDB không hợp lệ: {artifact}')
        actual_hash = sha_array(values)
        expected_hash = provenance['training_data_fingerprint'][f'{artifact.lower()}_noise']
        if actual_hash != expected_hash:
            raise ValueError(f'{artifact}: fingerprint khác NSTDB khi train; kiểm tra cache/nguồn dữ liệu.')
        noise[artifact] = values
    return clean, meta, noise, pd.DataFrame(audits)

CLEAN_RAW, CLEAN_META, NOISE, AUDIT = prepare_pilot_data(
    PROVENANCE, TEST_RECORD_NAMES, MAX_BEATS_PER_RECORD, CACHE_DIR)
display(AUDIT[['record_id', 'accepted_segments', 'selected_for_pilot']])
display(CLEAN_META[['sample_id', 'record_id', 'unpadded_length']].head(10))


## 5. Tạo các cặp noisy test

Đầu vào mô hình và ECG tham chiếu dùng chung hệ số `max(abs(clean))`, đúng giao thức nghiên cứu hiện tại. **Đây là chuẩn hóa dựa vào tham chiếu**, chưa phải quy tắc triển khai trên ECG thực chưa biết clean. Không chuẩn hóa riêng prediction để làm hình đẹp hơn. Dùng biên độ chuẩn hóa trên hình, không gắn nhãn mV.


In [ ]:
def make_pilot_pairs(clean_raw, clean_meta, noise, groups, level_r):
    allowed = ('BW', 'EM', 'MA', 'BW+EM', 'BW+MA', 'EM+MA', 'BW+EM+MA')
    if not groups or len(set(groups)) != len(groups) or not set(groups).issubset(allowed):
        raise ValueError('NOISE_GROUPS phải là các tổ hợp khác nhau thuộc MIX7.')
    if not np.isfinite(level_r) or not 0.2 <= level_r <= 1.99:
        raise ValueError('NOISE_LEVEL_R phải nằm trong [0.2, 1.99]. Đây không phải dB.')
    clean_list, noisy_list, rows = [], [], []
    for group in groups:
        for index, raw in enumerate(clean_raw):
            patches, starts = [], {}
            for artifact in group.split('+'):
                values = noise[artifact]
                midpoint = len(values) // 2
                # Protocol 1: opposite channel and second half, never the train patches.
                heldout = values[midpoint:, 1]
                count = len(heldout) // LENGTH
                if not count:
                    raise ValueError(f'{artifact}: nửa test quá ngắn.')
                start = (index % count) * LENGTH
                patches.append(heldout[start:start + LENGTH])
                starts[artifact] = midpoint + start
            mixture = np.sum(np.stack(patches), axis=0)
            if np.ptp(raw) <= 0 or np.ptp(mixture) <= 0:
                raise ValueError('Clean/nhiễu có biên độ bằng 0; không bỏ qua mẫu.')
            noisy_raw = raw + float(level_r) * float(np.ptp(raw)) / float(np.ptp(mixture)) * mixture
            scale = max(float(np.max(np.abs(raw))), 1e-8)
            clean_list.append((raw / scale).astype(np.float32))
            noisy_list.append((noisy_raw / scale).astype(np.float32))
            row = clean_meta.iloc[index].to_dict()
            row.update({
                'pair_id': f"{group}|{row['sample_id']}", 'noise_group': group,
                'split': 'test_pilot', 'protocol': 1, 'noise_channel_zero_based': 1,
                'noise_half': 'second', 'noise_start_json': json.dumps(starts, sort_keys=True),
                'noise_level_r': float(level_r), 'clean_maxabs_scale': scale,
                'sampling_rate': TARGET_FS, 'insert_offset': INSERT_OFFSET,
            })
            rows.append(row)
    clean, noisy = np.stack(clean_list), np.stack(noisy_list)
    manifest = pd.DataFrame(rows)
    if not manifest.pair_id.is_unique or clean.shape != noisy.shape:
        raise ValueError('Các cặp bị trùng hoặc khác shape.')
    if not np.isfinite(clean).all() or not np.isfinite(noisy).all():
        raise ValueError('Có giá trị không hữu hạn trong input.')
    return clean, noisy, manifest

CLEAN, NOISY, MANIFEST = make_pilot_pairs(
    CLEAN_RAW, CLEAN_META, NOISE, NOISE_GROUPS, NOISE_LEVEL_R)
print('Clean/noisy:', CLEAN.shape, NOISY.shape)
display(MANIFEST.groupby('noise_group', sort=False).size().rename('Số cặp pilot'))


## 6. Suy luận DDIM và metric

`DENOISED` là ECG **sau toàn bộ quá trình lấy mẫu**, không phải Gaussian do U-Net dự đoán. Không clamp đầu ra hoặc làm mượt hậu xử lý.

Lưu riêng metric **full512** (gồm padding) và **valid** (phần tín hiệu thực). Metric trên hình dùng **valid**, không trộn với số toàn đoạn. SNR và RMSE tính trên cùng cặp, cùng scale. Seed được đặt ngay trước suy luận. Kết quả pilot không thay thế benchmark chính thức.


In [ ]:
def prediction_metrics(clean, noisy, prediction, manifest):
    if clean.shape != noisy.shape or clean.shape != prediction.shape or not np.isfinite(prediction).all():
        raise ValueError('Prediction không hữu hạn hoặc không khớp input.')
    rows = []
    for i, meta in manifest.iterrows():
        row = meta.to_dict()
        regions = {'full512': slice(None), 'valid': slice(
            int(meta.insert_offset), int(meta.insert_offset + meta.unpadded_length))}
        for name, region in regions.items():
            x, y, p = [a[i, region].astype(np.float64) for a in (clean, noisy, prediction)]
            energy, err_in, err_out = np.sum(x*x), np.sum((x-y)**2), np.sum((x-p)**2)
            floor = 1e-12
            row[f'rmse_{name}'] = float(np.sqrt(np.mean((x-p)**2)))
            row[f'snr_in_db_{name}'] = float(10*np.log10(max(energy, floor)/max(err_in, floor)))
            row[f'snr_out_db_{name}'] = float(10*np.log10(max(energy, floor)/max(err_out, floor)))
            row[f'delta_snr_db_{name}'] = row[f'snr_out_db_{name}'] - row[f'snr_in_db_{name}']
        rows.append(row)
    return pd.DataFrame(rows)


def infer_pilot(model, clean, noisy, manifest, provenance, batch_size, steps, eta, shots, seed, device):
    if not 1 <= steps <= model.num_steps or batch_size < 1 or shots < 1 or eta < 0:
        raise ValueError('Cấu hình sampler không hợp lệ.')
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if device.type == 'cuda':
        torch.cuda.manual_seed_all(seed)
    outputs = []
    tick = time.perf_counter()
    with torch.inference_mode():
        for start in range(0, len(noisy), batch_size):
            batch = torch.from_numpy(noisy[start:start + batch_size, None, :]).to(device)
            pred = model.denoising(batch, use_ddim=True, ddim_steps=steps,
                                   ddim_eta=eta, num_shots=shots)
            if pred.shape != batch.shape or not torch.isfinite(pred).all():
                raise ValueError(f'Prediction không hợp lệ tại batch {start}.')
            outputs.append(pred[:, 0].cpu().numpy())
            print(f'DDIM: {min(start + batch_size, len(noisy))}/{len(noisy)} cặp')
    prediction = np.concatenate(outputs).astype(np.float32)
    tau = [int(np.floor((model.num_steps / steps**2) * i**2)) for i in range(steps+1)]
    provenance.update({
        'sampler': 'original_U4_quadratic_tau_DDIM', 'ddim_steps': steps,
        'ddim_eta': eta, 'shots': shots, 'nfe_per_sample': steps*shots,
        'tau': tau, 'inference_seed': seed, 'inference_batch_size': batch_size,
        'determinism_note': 'Fixed seed and batch/order; CPU/GPU or version changes may differ.',
        'pilot_pair_count': len(noisy), 'pilot_clean_count': manifest.sample_id.nunique(),
        'selected_records': list(manifest.record_id.unique()),
        'selection_rule': 'First accepted beats per specified held-out record, before inference.',
        'noise_level_r': float(manifest.noise_level_r.iloc[0]),
        'pilot_wall_seconds': time.perf_counter()-tick,
        'scope': 'qualitative held-out pilot, NOT the full-test benchmark',
        'clean_sha256': sha_array(clean), 'noisy_sha256': sha_array(noisy),
        'prediction_sha256': sha_array(prediction),
    })
    return prediction

DENOISED = infer_pilot(MODEL, CLEAN, NOISY, MANIFEST, PROVENANCE,
                       BATCH_SIZE, DDIM_STEPS, DDIM_ETA, NUM_SHOTS, INFERENCE_SEED, DEVICE)
METRICS = prediction_metrics(CLEAN, NOISY, DENOISED, MANIFEST)
METRICS['checkpoint_sha256'] = PROVENANCE['checkpoint_sha256']
METRICS['weights'] = WEIGHTS_KIND
print('Chỉ là thống kê của tập pilot nhỏ:')
display(METRICS.groupby('noise_group', sort=False)[[
    'rmse_valid', 'snr_in_db_valid', 'snr_out_db_valid', 'delta_snr_db_valid']].mean().round(4))


## 7. Hình 5 — bốn nhóm nhiễu, hiển thị trực tiếp

Mỗi hàng: waveform chồng tham chiếu–nhiễu–phục hồi; phóng to quanh đỉnh biên độ tuyệt đối của **tham chiếu** ±120 ms; sai số phục hồi trừ tham chiếu. Vùng zoom là quy tắc hình học, **không phải xác nhận vị trí sóng R hay đánh giá lâm sàng**.

Cùng thang biên độ giữa các hàng, cùng scale lỗi giữa các hàng. Loại padding khỏi trục thời gian nhưng **không loại khỏi input mô hình**. Mẫu mặc định được xác định trước theo thứ tự dữ liệu, không tự chọn mẫu có chất lượng tốt nhất.


In [ ]:
def make_waveform_figure(clean, noisy, prediction, metrics, sample_id, groups, grayscale=False):
    indices = []
    for group in groups:
        found = metrics.index[(metrics.sample_id == sample_id) & (metrics.noise_group == group)]
        if len(found) != 1:
            raise ValueError(f'Cần đúng một cặp: {sample_id}/{group}')
        indices.append(int(found[0]))
    fig, axes = plt.subplots(len(indices), 3, figsize=(10.2, 2.35*len(indices)),
                             squeeze=False, gridspec_kw={'width_ratios': [2.1, 1.15, 1.15]},
                             layout='constrained')
    colors = {'clean': '#222222', 'noisy': '#9b9b9b' if grayscale else '#b78d31',
              'pred': '#555555' if grayscale else '#156c9c'}
    valid_arrays, errors = [], []
    for i in indices:
        row = metrics.loc[i]
        sl = slice(int(row.insert_offset), int(row.insert_offset + row.unpadded_length))
        valid_arrays.extend([a[i, sl] for a in (clean, noisy, prediction)])
        errors.append(prediction[i, sl]-clean[i, sl])
    low = min(float(a.min()) for a in valid_arrays)
    high = max(float(a.max()) for a in valid_arrays)
    margin = max((high-low)*0.08, 0.05)
    error_lim = max(max(float(np.max(np.abs(e))) for e in errors)*1.1, 0.05)
    for r, i in enumerate(indices):
        row = metrics.loc[i]
        offset, count = int(row.insert_offset), int(row.unpadded_length)
        sl = slice(offset, offset+count)
        t = np.arange(count)/float(row.sampling_rate)
        x, y, p = [a[i, sl] for a in (clean, noisy, prediction)]
        # Zoom is anchored to reference absolute peak, NOT to the nicest prediction.
        anchor = int(np.argmax(np.abs(x)))
        half = int(round(0.12*row.sampling_rate))
        a, b = max(0, anchor-half), min(count, anchor+half+1)
        for ax in axes[r, :2]:
            ax.plot(t, y, color=colors['noisy'], lw=0.85, alpha=0.85, label='Noisy')
            ax.plot(t, x, color=colors['clean'], lw=1.1, label='Reference')
            ax.plot(t, p, color=colors['pred'], lw=1.0, ls='--', label='Restored')
            ax.set_ylim(low-margin, high+margin)
        axes[r, 0].axvspan(t[a], t[b-1], color='#777777', alpha=0.1, lw=0)
        axes[r, 1].set_xlim(t[a], t[b-1])
        axes[r, 2].plot(t, p-x, color=colors['pred'], lw=1)
        axes[r, 2].axhline(0, color='#666666', lw=0.65, ls=':')
        axes[r, 2].set_ylim(-error_lim, error_lim)
        axes[r, 0].set_ylabel(f"{row.noise_group}\nNormalized amplitude", fontsize=9)
        axes[r, 0].set_title(
            f"SNR in/out: {row.snr_in_db_valid:.2f}/{row.snr_out_db_valid:.2f} dB",
            fontsize=9, loc='left')
        axes[r, 1].set_title('Reference-peak zoom (±120 ms)', fontsize=8)
        axes[r, 2].set_title(f"Error: restored − reference\nRMSE = {row.rmse_valid:.4f}", fontsize=9)
        for ax in axes[r]:
            ax.set_xlabel('Time (s)', fontsize=9)
            ax.grid(alpha=0.15)
            ax.spines[['top', 'right']].set_visible(False)
            ax.tick_params(labelsize=8)
    axes[0, 0].legend(loc='upper right', fontsize=8, frameon=False, ncol=3)
    fig.suptitle(f"Held-out pilot | {sample_id} | r={metrics.loc[indices[0], 'noise_level_r']:.2f}\n"
                 'Valid region only (padding removed); normalized amplitude, not mV', fontsize=10)
    return fig


def show_figure(fig, dpi=150):
    stream = BytesIO()
    fig.savefig(stream, format='png', dpi=dpi, facecolor='white')
    display(Image(data=stream.getvalue()))
    plt.close(fig)

selected_sample = PREVIEW_SAMPLE_ID or str(CLEAN_META.sample_id.iloc[0])
show_figure(make_waveform_figure(CLEAN, NOISY, DENOISED, METRICS,
                               selected_sample, FIGURE_GROUPS, GRAYSCALE), PREVIEW_DPI)
display(METRICS.loc[METRICS.sample_id == selected_sample, [
    'pair_id', 'rmse_valid', 'snr_in_db_valid', 'snr_out_db_valid', 'delta_snr_db_valid']])


## 8. Kiểm tra riêng các tổ hợp trộn

Dùng cùng mẫu clean đang xem. Khi chỉnh giao diện, chỉ chạy lại mục 7 hoặc 8, **không cần chạy lại DDIM**. Nếu muốn xem một mã mẫu khác, sửa `PREVIEW_SAMPLE_ID` và chạy lại mục 7 (chạy lại cell tham số nếu thay biến ở đó), không chạy lại các bước tải và suy luận.


In [ ]:
mixed_groups = tuple(g for g in NOISE_GROUPS if '+' in g)
if mixed_groups:
    show_figure(make_waveform_figure(CLEAN, NOISY, DENOISED, METRICS,
                                   selected_sample, mixed_groups, GRAYSCALE), PREVIEW_DPI)


## 9. Xuất khi đã kiểm tra (mặc định tắt)

Bật `EXPORT_FILES=True` và chạy cell này để lưu **`5.png` (600 dpi), `5.svg`**, arrays clean/noisy/denoised, manifest + metric và thông tin checkpoint/sampler/seed. Không tự tải xuống. Nếu thay mẫu hiển thị, chạy lại mục 7 trước khi xuất. Ảnh chỉ chứa các ví dụ pilot, không tự trở thành bằng chứng toàn tập test.


In [ ]:
def export_results(directory, fig, clean, noisy, prediction, metrics, provenance):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    fig.savefig(directory / '5.png', dpi=600, facecolor='white')
    fig.savefig(directory / '5.svg', facecolor='white')
    np.savez_compressed(directory / 'fig05_predictions.npz', clean=clean,
                        noisy=noisy, denoised=prediction,
                        pair_id=metrics.pair_id.to_numpy(dtype=str))
    metrics.to_csv(directory / 'fig05_metrics_manifest.csv', index=False)
    (directory / 'fig05_provenance.json').write_text(
        json.dumps(provenance, ensure_ascii=False, indent=2), encoding='utf-8')
    print('Đã lưu (không tự tải xuống):', directory.resolve())

EXPORT_FILES = False
if EXPORT_FILES:
    PROVENANCE['figure_sample_id'] = selected_sample
    PROVENANCE['figure_noise_groups'] = list(FIGURE_GROUPS)
    PROVENANCE['figure_metric_region'] = 'valid, excluding zero padding'
    PROVENANCE['figure_zoom_rule'] = 'reference absolute peak +/-120 ms'
    figure = make_waveform_figure(CLEAN, NOISY, DENOISED, METRICS,
                                 selected_sample, FIGURE_GROUPS, GRAYSCALE)
    export_results(EXPORT_DIR, figure, CLEAN, NOISY, DENOISED, METRICS, PROVENANCE)
    plt.close(figure)
else:
    print('Chưa xuất file; hình đã hiển thị phía trên. Cache dữ liệu vẫn được giữ trong phiên.')


## 10. Đưa lên Overleaf sau khi duyệt

Tải `5.png` lên cùng workspace Overleaf và dùng tên file trực tiếp:

```latex
\includegraphics[width=\textwidth]{5.png}
```

Chưa tự sửa `paper.tex`: cần duyệt waveform và xác nhận bộ ví dụ trước. Khi viết caption, ghi mã mẫu, U4 MIX7 512, loại trọng số, cấu hình DDIM, chuẩn hóa và phạm vi pilot. Không diễn giải cải thiện SNR của vài đoạn thành kết luận toàn bộ tập test hoặc bảo toàn chẩn đoán.
